# Jedna partycja per device
Wszystkie dane urządzenia trafiają do jednej partycji, co sprawia, że odczyt staje się coraz wolniejszy.
Partition key = device_id

In [ ]:
import uuid
from cassandra.cluster import Cluster

In [ ]:
# Otwieranie połączenia
cluster = Cluster(['cassandra_nosql_lab'], port=9042)
session = cluster.connect()
print("Połączono z Cassandra")

In [ ]:
# Tworzenie Keyspace
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS lab_keyspace
    WITH replication = {'class': 'SimpleStrategy', 'replication_factor': '1'}
""")
print("Utworzono keyspace")

In [ ]:
# Użycie Keyspace
session.set_keyspace('lab_keyspace')

# Tworzenie tabeli
session.execute("""
    CREATE TABLE IF NOT EXISTS events_by_device_1 (
        device_id TEXT,
        day TEXT,
        event_time TIMESTAMP,
        temperature FLOAT,
        humidity FLOAT,
        PRIMARY KEY (device_id)
    )
""")
print("Utworzono tabelę 'users'")

## Generate test data

In [ ]:
!python /workspace/_data_generator/main.py \
  --loader cassandra \
  --table events_by_device_1 \
  --records 1000 \
  --batch-size 100

In [ ]:
rows = session.execute("""
SELECT *
FROM
    events_by_device_1
WHERE
    device_id='device_1'
LIMIT 10
""")
print("Znalezione rekordy:")
for row in rows:
    print(f"- {row.device_id} {row.day} ({row.temperature}°C, {row.humidity}%) – {row.event_time}")

In [ ]:
# Zamknięcie połączenia
cluster.shutdown()